# **ST 554 Spring 2026 Final Project**
## *Created by Cody Ashby on April 27, 2026*
### Fitting The Model

For this project, we are going to incorporate the concepts of fitting a machine learning model with `PySpark` from an SQL-style dataframe and applying numerous transformations along with reading and writing streamed data and producing our own streamed data with Spark Structured Streaming.

As always, we need to import a bunch of modules and other odds and ends...

In [1]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from pyspark.sql.functions import window, col
from pyspark.sql.types import StructType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, Binarizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder # set parallelism to 128 when doing CV!
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 22:19:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Use `pandas` to read in a dataset regarding power consumption from varying timezones in Tetouan City.

In [2]:
power_data = pd.read_csv("power_ml_data.csv")

Here, we convert this into a spark SQL-style dataframe; the schema for this dataframe is listed below.

In [3]:
spark_power_data = spark.read.csv("power_ml_data.csv",inferSchema=True,header=True)
spark_power_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

Now, we'll apply a large amount of transformations to fit a regularized MLR model. We'll begin with an `SQLTransformer`.

In [4]:
sqlTrans=SQLTransformer(statement="SELECT Month, Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, \
                                        Power_Zone_1, Power_Zone_2, Power_Zone_3 as label, CAST(Hour AS DOUBLE) AS Revised_Hour FROM __THIS__")

After casting the `Hour` variable as a DoubleType, we'll use the `Binarizer` to create a binary variable.

In [5]:
binary_Hour=Binarizer(threshold=6.5,inputCol="Revised_Hour",outputCol="Binary_Hour")

Since there are a couple of time-related variables in this dataset, we'll perform a one-hot encoding of the `Month` variable so that it doesn't get treated as a numeric variable, like a majority of the rest.

In [6]:
#Month_Index=StringIndexer(inputCol="Month",outputCol="Indexed_Month")
Month_OHE=OneHotEncoder(inputCol="Month",outputCol="Encoded_Month")

Another relatively neat thing we can do is perform a PCA (principal component analysis) on the weather variables. The top two components will be considered.

In [7]:
Weather_Vars=VectorAssembler(inputCols=['Temperature','Humidity','Wind_Speed','General_Diffuse_Flows','Diffuse_Flows'],outputCol='weather_features',handleInvalid='keep')
Weather_Comps=Weather_Vars.transform(spark_power_data)
PC_features=PCA(k=2,inputCol='weather_features',outputCol="Prin_Comps")
Fitted_PC=PC_features.fit(Weather_Comps)

All of these transformations above will be put into a `VectorAssembler` so we can create a pipeline for ease of computation.

In [8]:
total_features=VectorAssembler(inputCols=['Prin_Comps','Binary_Hour','Power_Zone_1','Power_Zone_2','Encoded_Month'],outputCol='features',handleInvalid='keep')

After defining an estimator, we'll construct our pipeline.

In [9]:
MLR=LinearRegression()
grand_pipeline=Pipeline(stages=[sqlTrans,binary_Hour,Month_OHE,Weather_Vars,PC_features,total_features,MLR])

Below is a grid of regularization parameters to test for when fitting the model.

In [10]:
EN_Param_Grid=ParamGridBuilder().addGrid(MLR.regParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]) \
                            .addGrid(MLR.elasticNetParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]).build()

We will now proceed with cross-validation to fit the model with our data. A `parallelism` parameter is used to speed up computation.

In [11]:
ElasticNet_CrossVal=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=EN_Param_Grid,
                                   evaluator=RegressionEvaluator(metricName='rmse'),
                                   numFolds=5,
                                   seed=8,
                                   parallelism=64)
ElasticNet_Model=ElasticNet_CrossVal.fit(spark_power_data)

Below is a list of all the RMSEs associated with each fitted model along with the pair of hyperparameters used.

In [12]:
ElasticNet_RMSEs=[]
for _ in range(len(EN_Param_Grid)):
    ElasticNet_RMSEs.append([ElasticNet_Model.avgMetrics[_],EN_Param_Grid[_].values()])
ElasticNet_RMSEs

[[2147.743822079609, dict_values([0.0, 0.0])],
 [2147.743822079609, dict_values([0.0, 0.05])],
 [2147.743822079609, dict_values([0.0, 0.1])],
 [2147.743822079609, dict_values([0.0, 0.25])],
 [2147.743822079609, dict_values([0.0, 0.5])],
 [2147.743822079609, dict_values([0.0, 0.75])],
 [2147.743822079609, dict_values([0.0, 0.9])],
 [2147.743822079609, dict_values([0.0, 0.95])],
 [2147.743822079609, dict_values([0.0, 0.98])],
 [2147.743822079609, dict_values([0.0, 0.99])],
 [2147.743822079609, dict_values([0.0, 1.0])],
 [2147.743804871119, dict_values([0.05, 0.0])],
 [2147.745520060856, dict_values([0.05, 0.05])],
 [2147.738868265524, dict_values([0.05, 0.1])],
 [2147.7435245465463, dict_values([0.05, 0.25])],
 [2147.743723303225, dict_values([0.05, 0.5])],
 [2147.7424631014997, dict_values([0.05, 0.75])],
 [2147.7427017257346, dict_values([0.05, 0.9])],
 [2147.74267120382, dict_values([0.05, 0.95])],
 [2147.742633940907, dict_values([0.05, 0.98])],
 [2147.742635289277, dict_values([0.05

Upon very close inspection of each RMSE associated with each pair of hyperparameters, we can see that the lowest RMSE goes to the one with the `regParam` of 0.05 and the `ElasticNetParam` of 0.1, indicating fairly minimal regularization required for this model. The CV error, as well as the training RMSE, for this model is shown below.

In [13]:
ElasticNet_CV_Error=np.mean(ElasticNet_Model.avgMetrics)
ElasticNet_Training_RMSE=min(ElasticNet_Model.avgMetrics)
print(ElasticNet_CV_Error)
print(ElasticNet_Training_RMSE)

2147.7574775784437
2147.738868265524


With those numbers in mind, we will now build our optimal Elastic Net model using the tuning parameters associated with the lowest RMSE.

In [14]:
Best_EN_Params=ParamGridBuilder().addGrid(MLR.regParam,[0.05]).addGrid(MLR.elasticNetParam,[0.1]).build()
Best_ElasticNet_CV=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=Best_EN_Params,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5)
Best_ElasticNet_Model=Best_ElasticNet_CV.fit(spark_power_data)

This fitted model will now be used to make predictions based on the original dataset. We'll also note how far each prediction is from the actual response with a `residuals` column, as shown below.

In [15]:
Best_ElasticNet_Model_Preds=Best_ElasticNet_Model.transform(spark_power_data)
Best_ElasticNet_Model_Preds_Resids=Best_ElasticNet_Model_Preds.withColumn("residuals",col("label")-col("prediction"))
Best_ElasticNet_Model_Preds_Resids.select("label","prediction","residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386| 20878.85066079531|-637.8868007953097|
|20131.08434|18660.227266546775| 1470.857073453226|
|19668.43373| 18204.75215311686|1463.6815768831402|
|18899.27711|17590.648498340925|1308.6286116590745|
|18442.40964|  16997.3019864581|1445.1076535419015|
|18130.12048|16517.686723495084|1612.4337565049173|
|17945.06024| 16093.24614105391| 1851.814098946088|
|17459.27711|15722.695360254002|1736.5817497459975|
|17025.54217|15271.043828662507| 1754.498341337494|
|16794.21687|14938.348764665025|1855.8681053349756|
|16638.07229|14652.383721423037| 1985.688568576963|
|16395.18072|14414.900662729378| 1980.280057270622|
|16117.59036|14082.889275685437|2034.7010843145636|
| 15822.6506|13624.882091434294|2197.7685085657067|
|15672.28916|13450.340775466415|2221.9483845335853|
|15597.10843| 13302.24639455996| 2294.862035440041|
|15510.36145

We'll use this concept above to make predictions on incoming data below.

### Streaming Data

The schema for the spark dataframe is placed into an object so we can read and write our stream.

In [16]:
spark_power_ml_schema=spark_power_data.schema

A folder has been created so we can read in files for our stream.

In [17]:
Read_Power_Stream=spark.readStream.schema(spark_power_ml_schema).csv("Other_Files",header=True)

Some modifications have been added to this stream so we can create a table for any incoming data. Note that the same stream was used twice for transformations and later joined together.

In [18]:
Streamed_Revised=Read_Power_Stream.withColumnRenamed("Power_Zone_3","label") \
                                .select("label","Temperature","Humidity","Wind_Speed","General_Diffuse_Flows", \
                                        "Diffuse_Flows","Power_Zone_1","Power_Zone_2","Month","Hour")
Streamed_Preds_Resids=Best_ElasticNet_Model.transform(Read_Power_Stream).withColumn("residual",col("label")-col("prediction")).select("label","prediction","residual")
Joined_Streams=Streamed_Preds_Resids.join(Streamed_Revised,"label","inner")

This joined stream will now be used to write a stream that allows us to see this table above. The query for this stream is defined below.

In [19]:
Power_Query=Joined_Streams.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17447.71084| 17667.24386214502|-219.53302214502037|       8.49|    76.9|     0.083|                0.084|        0.141| 28362.53165| 18320.97264|    1|   0|
|16029.58794|16727.599468623455| -698.0115286234559|      16.32|   52.15|     0.083|                603.7|        160.5| 32570.84746| 20677.20365|    2|  11|
|15381.29032|16410.605191945462| -1029.314871945462|      15.49|   63.14|     0.083|                324.0|       

### Producing Data

Now that the query has started, let's import the file that allows us to get several samples from the output. One strange thing I noticed was that despite the `.py` file specifying that five rows be sampled from the newer dataset, duplicates for each row appear in each batch.

In [20]:
!python3 Other_Files/PowerStreamDataProducer.py

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15350.78031|15324.531064925868|26.249245074131977|      10.67|    83.4|     0.086|                0.051|        0.156| 35370.34221| 29423.74962|   12|  21|
|15350.78031|15324.531064925868|26.249245074131977|      10.67|    83.4|     0.086|                0.051|        0.156| 35370.34221| 29423.74962|   12|  21|
|15350.78031|15324.531064925868|26.249245074131977|      10.67|    83.4|     0.086|                0.051|        0.156

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 12017.3494| 11643.20939973107|  374.1400002689297|       15.6|    85.3|      0.07|                0.037|        0.163| 24923.07692|  19033.8843|   11|   1|
| 12017.3494| 11643.20939973107|  374.1400002689297|       15.6|    85.3|      0.07|                0.037|        0.163| 24923.07692|  19033.8843|   11|   1|
| 12017.3494| 11643.20939973107|  374.1400002689297|       15.6|    85.3|      0.07|                0.037|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|22322.20633| 23775.98496143136|-1453.7786314313598|      22.58|   65.02|     0.269|                0.091|        0.089| 46398.58407| 28676.50728|    9|  20|
|22322.20633| 23775.98496143136|-1453.7786314313598|      22.58|   65.02|     0.269|                0.091|        0.089| 46398.58407| 28676.50728|    9|  20|
|22322.20633| 23775.98496143136|-1453.7786314313598|      22.58|   65.02|     0.269|                0.091|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17146.98795|17752.634284578722| -605.6463345787233|      15.55|   63.08|     4.921|                143.8|        154.1| 32433.41772| 19564.74164|    1|  17|
|17146.98795|17752.634284578722| -605.6463345787233|      15.55|   63.08|     4.921|                143.8|        154.1| 32433.41772| 19564.74164|    1|  17|
|17146.98795|17752.634284578722| -605.6463345787233|      15.55|   63.08|     4.921|                143.8|       

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|22004.81928| 20349.18771422039| 1655.6315657796113|      12.32|    78.0|      0.09|                0.044|        0.171| 34894.17722|  22821.8845|    1|  22|
|22004.81928| 20349.18771422039| 1655.6315657796113|      12.32|    78.0|      0.09|                0.044|        0.171| 34894.17722|  22821.8845|    1|  22|
|22004.81928| 20349.18771422039| 1655.6315657796113|      12.32|    78.0|      0.09|                0.044|       

...

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11473.36032| 13370.89035053226|-1897.5300305322598|      16.73|    76.2|     0.078|                1.767|         1.46| 22775.60656| 14377.70898|    5|   6|
|11473.36032| 13370.89035053226|-1897.5300305322598|      16.73|    76.2|     0.078|                1.767|         1.46| 22775.60656| 14377.70898|    5|   6|
|11473.36032| 13370.89035053226|-1897.5300305322598|      16.73|    76.2|     0.078|                1.767|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|30324.35146|28472.770878062016| 1851.5805819379857|      27.52|   69.04|     4.915|                401.7|        293.3| 38221.39535| 26065.82278|    7|  15|
|30324.35146|28472.770878062016| 1851.5805819379857|      32.28|   39.45|     4.922|                515.1|        66.23| 37998.13953| 25537.97468|    7|  17|
|30324.35146|28472.770878062016| 1851.5805819379857|      38.39|   16.94|     0.069|                418.7|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20927.80388| 21930.38209299401|-1002.5782129940089|      23.68|   46.02|     4.925|                0.069|          0.1| 44002.83186| 25020.37422|    9|  21|
|20927.80388| 21930.38209299401|-1002.5782129940089|      23.68|   46.02|     4.925|                0.069|          0.1| 44002.83186| 25020.37422|    9|  21|
|20927.80388| 21930.38209299401|-1002.5782129940089|      23.68|   46.02|     4.925|                0.069|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20690.52632| 19105.29555850451| 1585.2307614954916|      20.15|   54.84|     0.086|                0.084|        0.107| 34043.80328| 20530.03096|    5|  23|
|20690.52632| 19105.29555850451| 1585.2307614954916|      20.15|   54.84|     0.086|                0.084|        0.107| 34043.80328| 20530.03096|    5|  23|
|20690.52632| 19105.29555850451| 1585.2307614954916|      20.15|   54.84|     0.086|                0.084|      

-------------------------------------------
Batch: 20
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11420.89069|12931.989676953894|-1511.0989869538935|       18.9|   54.41|      0.08|                41.41|        35.14| 24815.21311| 16870.58824|    5|   7|
|11420.89069|12931.989676953894|-1511.0989869538935|       18.6|    81.3|     4.917|                12.33|        10.77| 24097.57377| 16220.43344|    5|   6|
|11420.89069|12931.989676953894|-1511.0989869538935|       18.9|   54.41|      0.08|                41.41|      